# CekAjaYuk - Real Dataset Preparation
## Persiapan Dataset Real untuk Training Model

Notebook ini menunjukkan cara menggunakan dataset gambar poster lowongan kerja real untuk training model yang optimal.

**PENTING**: Untuk hasil terbaik, gunakan dataset real dengan struktur:
```
dataset/
├── genuine/     # 500-1000+ poster lowongan ASLI
└── fake/        # 500-1000+ poster lowongan PALSU
```

In [ ]:
# Import required libraries
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print("\n⚠️  IMPORTANT: This notebook requires REAL dataset for optimal results!")
print("📁 Expected structure: dataset/genuine/ and dataset/fake/")

In [ ]:
# Configuration
DATASET_DIR = '../dataset'  # Path to real dataset
GENUINE_DIR = os.path.join(DATASET_DIR, 'genuine')
FAKE_DIR = os.path.join(DATASET_DIR, 'fake')
OUTPUT_DIR = '../data'
MODELS_DIR = '../models'

# Image processing settings
IMG_SIZE = (224, 224)
SUPPORTED_FORMATS = ['.jpg', '.jpeg', '.png', '.bmp']

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Dataset directory: {DATASET_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Check dataset availability
def check_dataset():
    """
    Check if real dataset is available
    """
    genuine_exists = os.path.exists(GENUINE_DIR)
    fake_exists = os.path.exists(FAKE_DIR)
    
    if genuine_exists and fake_exists:
        genuine_count = len([f for f in os.listdir(GENUINE_DIR) 
                           if any(f.lower().endswith(ext) for ext in SUPPORTED_FORMATS)])
        fake_count = len([f for f in os.listdir(FAKE_DIR) 
                        if any(f.lower().endswith(ext) for ext in SUPPORTED_FORMATS)])
        
        print(f"✅ Real dataset found!")
        print(f"   Genuine images: {genuine_count}")
        print(f"   Fake images: {fake_count}")
        print(f"   Total images: {genuine_count + fake_count}")
        
        if genuine_count < 100 or fake_count < 100:
            print(f"\n⚠️  WARNING: Recommended minimum 100 images per class")
            print(f"   Current: {genuine_count} genuine, {fake_count} fake")
        
        return True, genuine_count, fake_count
    else:
        print(f"❌ Real dataset not found!")
        print(f"   Expected: {GENUINE_DIR} and {FAKE_DIR}")
        print(f"\n📋 To use real dataset:")
        print(f"   1. Create folder: {DATASET_DIR}")
        print(f"   2. Add genuine job posters to: {GENUINE_DIR}")
        print(f"   3. Add fake job posters to: {FAKE_DIR}")
        print(f"   4. Re-run this notebook")
        
        return False, 0, 0

# Check dataset
dataset_available, genuine_count, fake_count = check_dataset()

In [ ]:
# Load real images
def load_real_images():
    """
    Load real job posting images from dataset
    """
    if not dataset_available:
        print("❌ Cannot load real dataset. Using synthetic data instead.")
        return create_synthetic_fallback()
    
    print("📂 Loading real images...")
    
    images = []
    labels = []
    filenames = []
    
    # Load genuine images
    print(f"   Loading genuine images from {GENUINE_DIR}...")
    for filename in os.listdir(GENUINE_DIR):
        if any(filename.lower().endswith(ext) for ext in SUPPORTED_FORMATS):
            img_path = os.path.join(GENUINE_DIR, filename)
            img = load_and_preprocess_image(img_path)
            if img is not None:
                images.append(img)
                labels.append(1)  # 1 for genuine
                filenames.append(filename)
    
    # Load fake images
    print(f"   Loading fake images from {FAKE_DIR}...")
    for filename in os.listdir(FAKE_DIR):
        if any(filename.lower().endswith(ext) for ext in SUPPORTED_FORMATS):
            img_path = os.path.join(FAKE_DIR, filename)
            img = load_and_preprocess_image(img_path)
            if img is not None:
                images.append(img)
                labels.append(0)  # 0 for fake
                filenames.append(filename)
    
    images = np.array(images)
    labels = np.array(labels)
    
    print(f"✅ Loaded {len(images)} images successfully")
    print(f"   Shape: {images.shape}")
    print(f"   Genuine: {np.sum(labels == 1)}")
    print(f"   Fake: {np.sum(labels == 0)}")
    
    return images, labels, filenames

def load_and_preprocess_image(img_path):
    """
    Load and preprocess single image
    """
    try:
        # Load image
        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️  Could not load: {img_path}")
            return None
        
        # Convert BGR to RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Resize to standard size
        img = cv2.resize(img, IMG_SIZE)
        
        # Normalize pixel values
        img = img.astype(np.float32) / 255.0
        
        return img
        
    except Exception as e:
        print(f"❌ Error processing {img_path}: {e}")
        return None

def create_synthetic_fallback():
    """
    Create synthetic data as fallback when real dataset is not available
    """
    print("🔄 Creating synthetic data as fallback...")
    
    # Create synthetic images
    n_samples = 1000
    images = np.random.rand(n_samples, IMG_SIZE[0], IMG_SIZE[1], 3)
    labels = np.random.randint(0, 2, n_samples)
    filenames = [f"synthetic_{i}.jpg" for i in range(n_samples)]
    
    print(f"⚠️  Using synthetic data: {n_samples} samples")
    print(f"   This is for DEMO purposes only!")
    print(f"   For real accuracy, use actual job posting images.")
    
    return images, labels, filenames

# Load images
X_images, y_labels, image_filenames = load_real_images()

In [ ]:
# Extract traditional ML features from real images
def extract_real_features(images):
    """
    Extract meaningful features from real job posting images
    """
    print("🔍 Extracting features from real images...")
    
    features_list = []
    feature_names = [
        'mean_red', 'mean_green', 'mean_blue',
        'std_red', 'std_green', 'std_blue',
        'brightness', 'contrast',
        'edge_density', 'texture_variance',
        'color_diversity', 'layout_symmetry'
    ]
    
    for i, img in enumerate(images):
        if i % 100 == 0:
            print(f"   Processing image {i+1}/{len(images)}...")
        
        # Convert to uint8 for OpenCV operations
        img_uint8 = (img * 255).astype(np.uint8)
        gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
        
        features = []
        
        # 1. Color features
        features.extend(np.mean(img, axis=(0, 1)))  # Mean RGB
        features.extend(np.std(img, axis=(0, 1)))   # Std RGB
        
        # 2. Brightness and contrast
        brightness = np.mean(gray)
        contrast = np.std(gray)
        features.extend([brightness, contrast])
        
        # 3. Edge density
        edges = cv2.Canny(gray, 50, 150)
        edge_density = np.sum(edges > 0) / (edges.shape[0] * edges.shape[1])
        features.append(edge_density)
        
        # 4. Texture variance
        texture_var = np.var(gray)
        features.append(texture_var)
        
        # 5. Color diversity (number of unique colors)
        unique_colors = len(np.unique(img_uint8.reshape(-1, 3), axis=0))
        color_diversity = unique_colors / (IMG_SIZE[0] * IMG_SIZE[1])
        features.append(color_diversity)
        
        # 6. Layout symmetry (simple horizontal symmetry measure)
        left_half = gray[:, :gray.shape[1]//2]
        right_half = np.fliplr(gray[:, gray.shape[1]//2:])
        if left_half.shape == right_half.shape:
            symmetry = 1 - np.mean(np.abs(left_half - right_half)) / 255
        else:
            symmetry = 0
        features.append(symmetry)
        
        features_list.append(features)
    
    features_array = np.array(features_list)
    
    print(f"✅ Feature extraction completed")
    print(f"   Features shape: {features_array.shape}")
    print(f"   Feature names: {feature_names}")
    
    return features_array, feature_names

# Extract features
X_features, feature_names = extract_real_features(X_images)

In [ ]:
# Split data and save
print("📊 Splitting data into train/validation/test sets...")

# Split images for deep learning
X_img_train, X_img_temp, y_img_train, y_img_temp = train_test_split(
    X_images, y_labels, test_size=0.3, random_state=42, stratify=y_labels
)

X_img_val, X_img_test, y_img_val, y_img_test = train_test_split(
    X_img_temp, y_img_temp, test_size=0.5, random_state=42, stratify=y_img_temp
)

# Split features for traditional ML
X_feat_train, X_feat_temp, y_feat_train, y_feat_temp = train_test_split(
    X_features, y_labels, test_size=0.3, random_state=42, stratify=y_labels
)

X_feat_val, X_feat_test, y_feat_val, y_feat_test = train_test_split(
    X_feat_temp, y_feat_temp, test_size=0.5, random_state=42, stratify=y_feat_temp
)

# Scale features
scaler = StandardScaler()
X_feat_train_scaled = scaler.fit_transform(X_feat_train)
X_feat_val_scaled = scaler.transform(X_feat_val)
X_feat_test_scaled = scaler.transform(X_feat_test)

print(f"Data split completed:")
print(f"  Training: {len(X_img_train)} samples")
print(f"  Validation: {len(X_img_val)} samples")
print(f"  Test: {len(X_img_test)} samples")

# Save processed data
print(f"\n💾 Saving processed data to {OUTPUT_DIR}...")

# Save images for deep learning
np.save(os.path.join(OUTPUT_DIR, 'X_images_train.npy'), X_img_train)
np.save(os.path.join(OUTPUT_DIR, 'X_images_val.npy'), X_img_val)
np.save(os.path.join(OUTPUT_DIR, 'X_images_test.npy'), X_img_test)
np.save(os.path.join(OUTPUT_DIR, 'y_images_train.npy'), y_img_train)
np.save(os.path.join(OUTPUT_DIR, 'y_images_val.npy'), y_img_val)
np.save(os.path.join(OUTPUT_DIR, 'y_images_test.npy'), y_img_test)

# Save features for traditional ML
np.save(os.path.join(OUTPUT_DIR, 'X_train.npy'), X_feat_train_scaled)
np.save(os.path.join(OUTPUT_DIR, 'X_val.npy'), X_feat_val_scaled)
np.save(os.path.join(OUTPUT_DIR, 'X_test.npy'), X_feat_test_scaled)
np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_feat_train)
np.save(os.path.join(OUTPUT_DIR, 'y_val.npy'), y_feat_val)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'), y_feat_test)

# Save scaler and metadata
import joblib
joblib.dump(scaler, os.path.join(MODELS_DIR, 'feature_scaler.pkl'))

# Save feature names
with open(os.path.join(OUTPUT_DIR, 'feature_names.txt'), 'w') as f:
    for name in feature_names:
        f.write(f"{name}\n")

# Save dataset info
dataset_info = {
    'total_samples': len(X_images),
    'genuine_samples': np.sum(y_labels == 1),
    'fake_samples': np.sum(y_labels == 0),
    'image_size': IMG_SIZE,
    'feature_count': len(feature_names),
    'train_samples': len(X_img_train),
    'val_samples': len(X_img_val),
    'test_samples': len(X_img_test),
    'dataset_type': 'real' if dataset_available else 'synthetic'
}

import json
with open(os.path.join(OUTPUT_DIR, 'dataset_info.json'), 'w') as f:
    json.dump(dataset_info, f, indent=2)

print(f"✅ All data saved successfully!")
print(f"\n📋 Dataset Summary:")
for key, value in dataset_info.items():
    print(f"   {key}: {value}")

if not dataset_available:
    print(f"\n⚠️  REMINDER: Using synthetic data for demo purposes")
    print(f"   For production use, collect real job posting images!")